# 9. Pace & outcomes: dump-ins

Compare **5v5 dump-ins** by preceding total possession pace and whether the dumping team establishes possession within **five seconds**.

## 9.1. Load inputs

In [1]:
from pathlib import Path

import numpy as np
import pandas as pd
import plotly.graph_objects as go

PROCESSED_DATA_DIR = Path("../data/processed")
OUTCOME_SECONDS = 5

# Dumped zone entries distinguish dump-ins from defensive clearances.
events = pd.read_parquet(PROCESSED_DATA_DIR / "events_augmented.parquet").sort_values(
    "event_id"
)
movements = pd.read_parquet(PROCESSED_DATA_DIR / "pace_transitions.parquet")
dump_ins = events.loc[
    events["event"].eq("Zone Entry")
    & events["detail_1"].eq("Dumped")
    & events["is_5v5"]
].copy()

## 9.2. Calculate preceding total pace

Pool same-possession movements through the approach ending at the dump-in.

In [2]:
def add_dump_in_pace(dump_events, transition_events):
    movement_groups = {
        possession_id: group
        for possession_id, group in transition_events.groupby(
            "possession_id", sort=False
        )
    }
    rows = []
    for dump_in in dump_events.itertuples(index=False):
        history = movement_groups.get(dump_in.possession_id, transition_events.iloc[:0])
        # Include arrival at the dump-in release, but never subsequent movement.
        history = history.loc[history["end_event_id"].le(dump_in.event_id)]
        distance = float(history["distance_ft"].sum())
        duration = float(history["modeled_elapsed_seconds"].sum())
        rows.append(
            {
                "event_id": dump_in.event_id,
                "preceding_transitions": len(history),
                "preceding_distance_ft": distance,
                "preceding_movement_seconds": duration,
                "preceding_total_pace_ft_s": distance / duration
                if duration > 0
                else np.nan,
            }
        )
    return dump_events.merge(pd.DataFrame(rows), on="event_id", validate="one_to_one")


dump_ins = add_dump_in_pace(dump_ins, movements)

## 9.3. Identify recovery within five seconds

Success is dumping-team control within five seconds, even after opponent control.

In [3]:
CONTROL_EVENTS = {
    "Puck Recovery",
    "Takeaway",
    "Pass",
    "Pass Reception",
    "Incomplete Pass",
    "Shot",
    "Goal",
    "Dump In/Out",
    "Zone Entry",
}
RESTART_EVENTS = {"Faceoff Win", "Penalty Shot Goal"}


def add_dump_in_outcomes(dump_events, all_events):
    period_events = {
        key: group
        for key, group in all_events.groupby(["game_id", "period"], sort=False)
    }
    rows = []
    for dump_in in dump_events.itertuples(index=False):
        period = period_events[(dump_in.game_id, dump_in.period)]
        complete_followup = (
            dump_in.modeled_clock_seconds - period["modeled_clock_seconds"].min()
            >= OUTCOME_SECONDS
        )
        following = period.loc[
            period["event_id"].gt(dump_in.event_id)
            & period["modeled_clock_seconds"].between(
                dump_in.modeled_clock_seconds - OUTCOME_SECONDS,
                dump_in.modeled_clock_seconds,
            )
        ]
        outcome = "Unresolved"
        reason = "Incomplete follow-up"
        control_event_id = None
        for event in following.itertuples(index=False):
            if event.event in RESTART_EVENTS:
                reason = "Restart"
                break
            if not event.is_5v5:
                reason = "Manpower change"
                break
            if event.event in CONTROL_EVENTS:
                if event.team_id not in {dump_in.team_id, dump_in.opponent_team_id}:
                    raise ValueError(
                        "Control event is not credited to a participating team."
                    )
                # Opponent control does not end the recovery window.
                if event.team_id == dump_in.team_id:
                    outcome = "Dumping team"
                    reason = "Team control within window"
                    control_event_id = event.event_id
                    break
        else:
            if complete_followup:
                outcome = "No team recovery"
                reason = "Complete five-second window"
        rows.append(
            {
                "event_id": dump_in.event_id,
                "recovery_outcome": outcome,
                "outcome_reason": reason,
                "control_event_id": control_event_id,
            }
        )
    return dump_events.merge(pd.DataFrame(rows), on="event_id", validate="one_to_one")


dump_ins = add_dump_in_outcomes(dump_ins, events)

## 9.4. Summarize quartiles

In [4]:
def summarize_dump_ins(data):
    eligible = data.loc[data["preceding_total_pace_ft_s"].notna()].copy()
    eligible["team_recovery"] = eligible["recovery_outcome"].eq("Dumping team")
    eligible["no_team_recovery"] = eligible["recovery_outcome"].eq("No team recovery")
    eligible["unresolved"] = eligible["recovery_outcome"].eq("Unresolved")
    bands, edges = pd.qcut(
        eligible["preceding_total_pace_ft_s"],
        q=4,
        labels=False,
        retbins=True,
        duplicates="drop",
    )
    eligible["quartile"] = bands + 1
    boundaries = pd.DataFrame(
        {
            "quartile": np.arange(1, len(edges)),
            "lower_pace_ft_s": edges[:-1],
            "upper_pace_ft_s": edges[1:],
        }
    )
    summary = (
        eligible.loc[eligible["quartile"].notna()]
        .groupby("quartile", observed=True)
        .agg(
            dump_ins=("event_id", "size"),
            team_recoveries=("team_recovery", "sum"),
            no_team_recovery=("no_team_recovery", "sum"),
            unresolved=("unresolved", "sum"),
            games=("game_id", "nunique"),
            median_pace_ft_s=("preceding_total_pace_ft_s", "median"),
        )
        .reset_index()
    )
    for count, percentage in [
        ("team_recoveries", "team_recovery_pct"),
        ("no_team_recovery", "no_team_recovery_pct"),
        ("unresolved", "unresolved_pct"),
    ]:
        summary[percentage] = 100 * summary[count] / summary["dump_ins"]
    return summary.merge(boundaries, on="quartile", validate="one_to_one"), boundaries


dump_in_summary, quartile_boundaries = summarize_dump_ins(dump_ins)
coverage_df = (
    dump_ins.assign(
        missing_pace=dump_ins["preceding_total_pace_ft_s"].isna(),
        zero_pace=dump_ins["preceding_total_pace_ft_s"].eq(0),
        measurable=dump_ins["preceding_total_pace_ft_s"].notna(),
    )
    .groupby("recovery_outcome")
    .agg(
        dump_ins=("event_id", "size"),
        missing_pace=("missing_pace", "sum"),
        zero_pace=("zero_pace", "sum"),
        measurable=("measurable", "sum"),
    )
    .reset_index()
)
display(coverage_df)
# Keep the reasons for unresolved cases visible.
display(
    dump_ins.loc[dump_ins["recovery_outcome"].eq("Unresolved")]
    .groupby("outcome_reason")
    .size()
    .rename("dump_ins")
    .reset_index()
)

,recovery_outcome,dump_ins,missing_pace,zero_pace,measurable
0,Dumping team,327,9,7,318
1,No team recovery,891,49,8,842
2,Unresolved,57,1,0,56


,outcome_reason,dump_ins
0,Incomplete follow-up,2
1,Manpower change,1
2,Restart,54


## 9.5. Display dump-in outcomes

In [5]:
def dump_in_outcomes_figure(summary, boundaries):
    outcomes = [
        ("team_recovery_pct", "Team recovery within 5 s", "#2563eb", "white"),
        ("no_team_recovery_pct", "No team recovery within 5 s", "#cbd5e1", "#334155"),
    ]
    bars = summary.set_index("quartile").reindex(range(1, len(boundaries) + 1))
    customdata = bars[
        [
            "dump_ins",
            "team_recoveries",
            "no_team_recovery",
            "unresolved",
            "games",
            "lower_pace_ft_s",
            "upper_pace_ft_s",
            "median_pace_ft_s",
        ]
    ].to_numpy()
    figure = go.Figure()
    for metric, label, color, text_color in outcomes:
        figure.add_trace(
            go.Bar(
                x=[f"Q{q}" for q in bars.index],
                y=bars[metric],
                name=label,
                marker_color=color,
                insidetextfont=dict(color=text_color),
                text=[
                    f"{value:.1f}%" if pd.notna(value) and value > 0 else ""
                    for value in bars[metric]
                ],
                textposition="inside",
                insidetextanchor="middle",
                customdata=customdata,
                hovertemplate=(
                    f"%{{x}}<br>{label}: %{{y:.1f}}%"
                    "<br>Dump-ins: %{customdata[0]:.0f}"
                    "<br>Team recovery within 5 s: %{customdata[1]:.0f}"
                    "<br>No team recovery within 5 s: %{customdata[2]:.0f}"
                    "<br>Unresolved: %{customdata[3]:.0f}"
                    "<br>Games: %{customdata[4]:.0f}"
                    "<br>Pace range: %{customdata[5]:.2f} to %{customdata[6]:.2f} ft/s"
                    "<br>Median pace: %{customdata[7]:.1f} ft/s<extra></extra>"
                ),
            )
        )
    if summary.empty:
        note = "No data: no measurable pace with assigned bands."
    elif len(boundaries) < 4:
        note = f"Only {len(boundaries)} bands: ties prevent four quartiles."
    else:
        note = None
    if note:
        figure.add_annotation(
            text=note, x=0.5, y=-0.30, xref="paper", yref="paper", showarrow=False
        )
    figure.update_layout(
        title="Total pace before dump-in | 5v5 | recovery within 5 seconds",
        template="plotly_white",
        barmode="stack",
        height=500,
        legend=dict(orientation="h", x=0.5, xanchor="center", y=1.05),
        margin=dict(t=100, b=100),
    )
    figure.update_xaxes(
        title_text="Preceding total possession pace quartile",
        categoryorder="array",
        categoryarray=["Q1", "Q2", "Q3", "Q4"],
    )
    figure.update_yaxes(range=[0, 100], title_text="Dump-ins (%)")
    return figure


dump_in_figure = dump_in_outcomes_figure(dump_in_summary, quartile_boundaries)
dump_in_figure.show()

## 9.6. Export dump-in outcomes

In [6]:
dump_in_outcomes_df = (
    dump_in_summary[
        [
            "quartile",
            "dump_ins",
            "team_recoveries",
            "no_team_recovery",
            "unresolved",
            "games",
            "median_pace_ft_s",
            "team_recovery_pct",
            "no_team_recovery_pct",
            "unresolved_pct",
            "lower_pace_ft_s",
            "upper_pace_ft_s",
        ]
    ]
    .astype(
        {
            "quartile": "Int64",
            "dump_ins": "Int64",
            "team_recoveries": "Int64",
            "no_team_recovery": "Int64",
            "unresolved": "Int64",
            "games": "Int64",
            "median_pace_ft_s": "Float64",
            "team_recovery_pct": "Float64",
            "no_team_recovery_pct": "Float64",
            "unresolved_pct": "Float64",
            "lower_pace_ft_s": "Float64",
            "upper_pace_ft_s": "Float64",
        }
    )
    .sort_values("quartile")
    .reset_index(drop=True)
)

PROCESSED_DATA_DIR.mkdir(parents=True, exist_ok=True)
export_path = PROCESSED_DATA_DIR / "dump_in_outcomes.parquet"
dump_in_outcomes_df.to_parquet(
    export_path, engine="pyarrow", compression="snappy", index=False
)
saved = pd.read_parquet(export_path)
pd.testing.assert_frame_equal(saved, dump_in_outcomes_df, check_exact=True)
print(f"Exported {len(saved):,} rows and {len(saved.columns)} columns to {export_path}")
display(saved)

Exported 4 rows and 12 columns to ..\data\processed\dump_in_outcomes.parquet


,quartile,dump_ins,team_recoveries,no_team_recovery,unresolved,games,median_pace_ft_s,team_recovery_pct,no_team_recovery_pct,unresolved_pct,lower_pace_ft_s,upper_pace_ft_s
0,1,304,65,223,16,34,11.45614,21.381579,73.355263,5.263158,0.0,14.899138
1,2,304,82,211,11,34,17.089793,26.973684,69.407895,3.618421,14.899138,19.30746
2,3,304,95,193,16,34,21.249237,31.25,63.486842,5.263158,19.30746,23.705321
3,4,304,76,215,13,34,27.537315,25.0,70.723684,4.276316,23.705321,108.9379
